In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-google-genai
!pip install pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.6 MB/s eta 0:00:00


In [2]:
from enum import Enum
from typing import List

from pydantic import BaseModel, Field


class Sentiment(str, Enum):
    positive = "positive"
    neutral = "neutral"
    negative = "negative"
    mixed = "mixed"


class Severity(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"
    critical = "critical"

In [3]:
class TaxonomyLabel(str, Enum):
    TUTOR_DELIVERY = "Tutor|Delivery"
    TUTOR_COMMUNICATION = "Tutor|Communication"
    TUTOR_KNOWLEDGE = "Tutor|Knowledge"
    TUTOR_PACE = "Tutor|Pace"

    CONTENT_DIFFICULTY = "Content|Difficulty"
    CONTENT_RELEVANCE = "Content|Relevance"
    CONTENT_CLARITY = "Content|Clarity"
    CONTENT_PACE = "Content|Pace"

    ASSESSMENT_FAIRNESS = "Assessment|Fairness"
    ASSESSMENT_DIFFICULTY = "Assessment|Difficulty"
    ASSESSMENT_ALIGNMENT = "Assessment|Alignment"

    ADMIN_SCHEDULING = "Administration|Scheduling"
    ADMIN_COMMUNICATION = "Administration|Communication"

    TECH_CONTENT = "Tech|Content"
    TECH_ASSESSMENT = "Tech|Assessment"
    TECH_PLATFORM = "Tech|Platform"
    TECH_MEDIA = "Tech|Media"
    TECH_UX = "Tech|UX"
    TECH_INFRA = "Tech|Infra"

    UNKNOWN = "Unknown|Unknown"

In [4]:
class FeedbackLabel(BaseModel):
    label: TaxonomyLabel
    sentiment: Sentiment
    severity: Severity
    confidence: float = Field(ge=0, le=1)

In [5]:
class FeedbackClassification(BaseModel):
    actionable: bool = Field(
        ...,
        description="Whether the feedback requires action or investigation"
    )

    summary: str = Field(
        ...,
        description="Short 1-2 sentence summary of the feedback"
    )

    labels: List[FeedbackLabel] = Field(
        default_factory=list,
        description="One or more classified topics"
    )

In [6]:
EXAMPLES = [
    {
        "input": "The instructor explains concepts clearly and uses excellent examples.",
        "output": {
            "actionable": False,
            "summary": "Positive feedback on teaching quality.",
            "labels": [
                {
                    "category": "Tutor",
                    "sub_category": "Delivery",
                    "sentiment": "Positive",
                    "severity": "Low",
                    "confidence": 0.98
                }
            ]
        }
    },
    {
        "input": "Auto grader keeps giving zero even when answers are correct.",
        "output": {
            "actionable": True,
            "summary": "Auto grader incorrectly scores correct answers.",
            "labels": [
                {
                    "category": "Tech",
                    "sub_category": "Assessment",
                    "sentiment": "Negative",
                    "severity": "High",
                    "confidence": 0.97
                }
            ]
        }
    }
]

In [28]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI


def get_model(provider="openai"):

    if provider == "openai":
        return ChatOpenAI(
            model="gpt-5-mini",
            temperature=0,
            api_key=userdata.get('OPENAI_API_KEY')
        )

    elif provider == "gemini":
        return ChatGoogleGenerativeAI(
            model="gemini-3.6-flash",
            temperature=0,
            api_key=userdata.get('GOOGLE_API_KEY')
        )

    else:
        raise ValueError("Unknown provider")

In [8]:
SYSTEM_PROMPT = """
You are a course feedback classification engine.

Your task is to analyze student feedback and produce structured JSON.

The objective is to identify actionable issues and route them to the correct team.

You must strictly follow the taxonomy provided below.

# Taxonomy

## Tutor

### Delivery

Quality of teaching, explanations, examples, demonstrations, presentation style.

Examples:

* Explanations are clear.
* Instructor uses good examples.
* Teaching style is confusing.

### Communication

Language clarity, speaking ability, responsiveness, interaction.

Examples:

* Hard to understand instructor.
* Tutor responds quickly.
* Communication is unclear.

### Knowledge

Subject expertise and accuracy.

Examples:

* Tutor is highly knowledgeable.
* Instructor gives incorrect information.

### Pace

Speed of teaching.

Examples:

* Too fast.
* Too slow.

---

## Content

### Difficulty

Content too easy or too difficult.

### Relevance

Content matches learning objectives and industry needs.

### Clarity

Content organization and understandability.

### Pace

Amount of content covered per unit or week.

---

## Assessment

### Fairness

Grading fairness and consistency.

### Difficulty

Assessment complexity.

### Alignment

Assessment matches taught material.

---

## Administration

### Scheduling

Timetable, deadlines, release dates.

### Communication

Administrative announcements and notifications.

---

## Tech

### Content

Broken learning materials, incorrect resources, inaccessible files.

### Assessment

Auto-grader issues, quiz bugs, submission problems.

### Platform

Login issues, crashes, outages, loading problems.

### Media

Audio, video, captions, streaming quality.

### UX

Navigation, usability, discoverability.

### Infra

Performance, servers, networking, lab environments.

---

## Unknown

Use only when feedback cannot reasonably fit any category.

# Actionability Rules

Mark actionable=false when feedback is:

1. Pure praise without improvement opportunity.
2. Pure personal preference.
3. Too vague to determine an action.
4. Not related to the course experience.

Examples:

"Great course."
→ actionable=false

"I personally hate Python."
→ actionable=false

"Nice."
→ actionable=false

# Multi-label Rules

A feedback may belong to multiple categories.

Example:

"The tutor explains well but the assignments are unfair."

Produces:

1. Tutor → Delivery → Positive
2. Assessment → Fairness → Negative

# Sentiment

Allowed values:

* Positive
* Negative
* Neutral
* Mixed

# Severity

Allowed values:

* Critical
* High
* Medium
* Low

Severity should reflect business impact.

Critical:

* Prevents learning
* Prevents assessment completion
* Platform unavailable

High:

* Major learning obstacle
* Repeated grading failures
* Significant course quality issue

Medium:

* Noticeable but not blocking

Low:

* Minor inconvenience

# Summary

Generate a concise summary of the core issue in less than 20 words.

# Confidence

Return a decimal value between 0 and 1.

# Output Requirements

Return JSON only.

Never explain your reasoning.

Never invent new categories.

Never invent new subcategories.

Use only categories from the taxonomy.

Use the exact schema provided.
"""

In [9]:
# feedback = "Well presented course material that takes you step by step through the required learning."

In [31]:
feedback = "very confusing and doesn't seem connected or flow"

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """
Example:
Input: {example_input}
Output: {example_output}

Now classify:

{feedback}
""")
])

In [29]:
llm = get_model("gemini")

structured_llm = llm.with_structured_output(
    FeedbackClassification
)

chain = prompt | structured_llm

In [32]:
result = chain.invoke(
    {
        "example_input" : EXAMPLES[0]["input"],
        "example_output" : EXAMPLES[0]["output"],
        "feedback": feedback,

    }
)

print(result.model_dump())

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'actionable': True, 'summary': 'Course content is confusing and lacks flow and logical connection.', 'labels': [{'label': <TaxonomyLabel.CONTENT_CLARITY: 'Content|Clarity'>, 'sentiment': <Sentiment.negative: 'negative'>, 'severity': <Severity.medium: 'medium'>, 'confidence': 0.92}]}


In [18]:
EXAMPLES[0]["output"]

{'actionable': False,
 'summary': 'Positive feedback on teaching quality.',
 'labels': [{'category': 'Tutor',
   'sub_category': 'Delivery',
   'sentiment': 'Positive',
   'severity': 'Low',
   'confidence': 0.98}]}